# Demo chatbota geoinformatycznego (Gemini 2.5 Flash)

Notebook demonstruje działanie klasy `ChatbotGemini` z pakietu `rdzen`.
Pokazuje: integrację z modelem, prompt systemowy, historię rozmowy,
kontrolę kontekstu liczonej w tokenach, parametry generacji oraz obsługę błędów.

**Wymagania:** plik `api_key.txt` w katalogu repozytorium lub zmienna `GEMINI_API_KEY`.

In [ ]:
# Instalacja zależności (jeśli potrzebna)
# !pip install -q google-genai python-dotenv

In [ ]:
import sys, os
# Dodajemy katalog repozytorium do ścieżki, by zaimportować pakiet rdzen
sys.path.insert(0, os.path.abspath('..'))
from rdzen import ChatbotGemini, Ustawienia, BladPolaczenia, LimitKontekstuPrzekroczony

## 1. Podstawowa rozmowa

Tworzymy chatbota z domyślnymi ustawieniami i zadajemy kilka pytań z domeny geoinformatyki.

In [ ]:
bot = ChatbotGemini()
print(bot.odpowiedz('Czym jest teledetekcja? Odpowiedz w 2 zdaniach.'))

## 2. Pamięć rozmowy (historia)

Model pamięta wcześniejsze wypowiedzi, bo do każdego zapytania dołączamy całą historię.
Poniższe pytanie odwołuje się do poprzedniej odpowiedzi bez powtarzania kontekstu.

In [ ]:
print(bot.odpowiedz('Podaj dwa przykłady sensorów wykorzystywanych w tej dziedzinie.'))
print('\n--- Liczba wiadomości w historii:', len(bot.historia.wiadomosci))
print('--- Tokeny kontekstu:', bot.historia.liczba_tokenow())

## 3. Parametry generacji

Niska temperatura → odpowiedzi bardziej deterministyczne i rzeczowe.
Wysoka temperatura → bardziej zróżnicowane, kreatywne. Porównajmy.

In [ ]:
for t in (0.1, 1.0):
    b = ChatbotGemini(Ustawienia(temperatura=t, maks_tokenow_odpowiedzi=80))
    print(f'--- temperatura={t} ---')
    print(b.odpowiedz('Zaproponuj nazwę dla aplikacji GIS do monitoringu lasów.'))
    print()

## 4. Kontrola długości kontekstu (w tokenach)

Ustawiamy mały budżet tokenów. Po kilku turach najstarsze wiadomości są usuwane,
ale prompt systemowy pozostaje nienaruszony.

In [ ]:
maly = ChatbotGemini(Ustawienia(budzet_tokenow_kontekstu=230, maks_tokenow_odpowiedzi=90))
for p in ['Czym jest NDVI? Krótko.', 'Jaki ma zakres? Krótko.',
          'Przykład w rolnictwie? Krótko.', 'Czym różni się od EVI? Krótko.']:
    maly.odpowiedz(p)
    print(f'po pytaniu "{p[:25]}..." -> wiadomości w pamięci: {len(maly.historia.wiadomosci)},'
          f' tokeny: {maly.historia.liczba_tokenow()}')

## 5. Obsługa błędów

Przy błędnym kluczu API system ponawia próby, a następnie zgłasza `BladPolaczenia`
i **wycofuje** wiadomość użytkownika, aby historia pozostała spójna.

In [ ]:
zly = ChatbotGemini(Ustawienia(klucz_api='ZLY_KLUCZ', liczba_prob=2, opoznienie_ponowienia_s=0.2))
try:
    zly.odpowiedz('test')
except BladPolaczenia as e:
    print('Złapano BladPolaczenia:', str(e)[:60], '...')
print('Historia po błędzie:', len(zly.historia.wiadomosci), 'wiadomości (spójna)')

## 6. Logi

Zapytania trafiają do `logi/zapytania.log`, a błędy do `logi/bledy.log`.

In [ ]:
print(open('../logi/zapytania.log', encoding='utf-8').read()[-600:])